In [1]:
import os
print(os.getcwd())
print(os.listdir())

C:\Users\PC\NICAI_project
['.ipynb_checkpoints', '.qodo', 'clean_weather.csv', 'integration_adapter.py', 'main.ipynb', 'nicai-validation-layer', 'REVIEW_PACKET.md', 'sanskar_engine.py']


In [2]:
import pandas as pd

df = pd.read_csv("clean_weather.csv")

print(df.head())
print(df.columns)

         date       city  temperature  aqi
0  2024-01-01     Mumbai         25.2  259
1  2024-01-01      Delhi         18.1  130
2  2024-01-01  Bengaluru         34.0   54
3  2024-01-01    Chennai         33.8  176
4  2024-01-01    Kolkata         22.5   97
Index(['date', 'city', 'temperature', 'aqi'], dtype='object')


In [3]:
print(df.columns)


Index(['date', 'city', 'temperature', 'aqi'], dtype='object')


CONVERT DATE + SORT

In [4]:
df["date"] = pd.to_datetime(df["date"])

df = df.sort_values(by=["city", "date"])

CREATE PREVIOUS VALUES

In [5]:
df["temp_prev"] = df.groupby("city")["temperature"].shift(1)
df["aqi_prev"] = df.groupby("city")["aqi"].shift(1)

trend Function

In [6]:
def compute_trend(row):
    if pd.isna(row["temp_prev"]) or pd.isna(row["aqi_prev"]):
        return 0.5

    temp_change = row["temperature"] - row["temp_prev"]
    aqi_change = row["aqi"] - row["aqi_prev"]

    # prioritize AQI (stronger signal)
    if aqi_change > 5:
        return 0.8   # RISING
    elif aqi_change < -5:
        return 0.2   # FALLING

    # fallback to temperature
    if temp_change > 1:
        return 0.8
    elif temp_change < -1:
        return 0.2

    return 0.5

apply trend

In [7]:
df["trend"] = df.apply(compute_trend, axis=1)

df.head()

,date,city,temperature,aqi,temp_prev,aqi_prev,trend
6,2024-01-01,Ahmedabad,28.9,345,NaN,NaN,0.5
16,2024-01-02,Ahmedabad,26.7,223,28.9,345.0,0.2
26,2024-01-03,Ahmedabad,29.8,131,26.7,223.0,0.2
36,2024-01-04,Ahmedabad,27.0,61,29.8,131.0,0.2
46,2024-01-05,Ahmedabad,21.6,338,27.0,61.0,0.8


In [8]:
from sanskar_engine import SanskarEngine

engine = SanskarEngine()

**Mapping Function**

In [9]:
def map_row_to_signals(row):
    return {
        "temperature": float(row["temperature"]),
        "pollution": float(row["aqi"]),
        "trend": float(row["trend"]),
        "zone": [row["city"]]
    }

RUN ENGINE ON DATA

In [10]:
from sanskar_engine import SanskarEngine

engine = SanskarEngine()

results = []

for _, row in df.iterrows():
    signals = map_row_to_signals(row)
    output = engine.process(signals)
    results.append(output)

print(results[:5])

[{'risk_level': 'HIGH', 'anomaly_type': 'severe_pollution', 'temporal_context': 'STABLE', 'spatial_context': 'Ahmedabad', 'confidence': 'HIGH', 'explanation': 'severe_pollution detected in Ahmedabad with STABLE trend due to high pollution levels.', 'recommendation_signal': 'severe_pollution'}, {'risk_level': 'MEDIUM', 'anomaly_type': 'high_pollution', 'temporal_context': 'FALLING', 'spatial_context': 'Ahmedabad', 'confidence': 'HIGH', 'explanation': 'high_pollution detected in Ahmedabad with FALLING trend due to high pollution levels.', 'recommendation_signal': 'high_pollution'}, {'risk_level': 'LOW', 'anomaly_type': 'normal', 'temporal_context': 'FALLING', 'spatial_context': 'Ahmedabad', 'confidence': 'LOW', 'explanation': 'normal detected in Ahmedabad with FALLING trend due to normal conditions.', 'recommendation_signal': 'normal'}, {'risk_level': 'LOW', 'anomaly_type': 'normal', 'temporal_context': 'FALLING', 'spatial_context': 'Ahmedabad', 'confidence': 'LOW', 'explanation': 'norma

In [11]:
df[["city", "date", "temperature", "aqi", "temp_prev", "aqi_prev", "trend"]].head(10)

,city,date,temperature,aqi,temp_prev,aqi_prev,trend
6,Ahmedabad,2024-01-01,28.9,345,NaN,NaN,0.5
16,Ahmedabad,2024-01-02,26.7,223,28.9,345.0,0.2
26,Ahmedabad,2024-01-03,29.8,131,26.7,223.0,0.2
36,Ahmedabad,2024-01-04,27.0,61,29.8,131.0,0.2
46,Ahmedabad,2024-01-05,21.6,338,27.0,61.0,0.8
56,Ahmedabad,2024-01-06,35.5,147,21.6,338.0,0.2
66,Ahmedabad,2024-01-07,24.7,128,35.5,147.0,0.2
76,Ahmedabad,2024-01-08,24.0,294,24.7,128.0,0.8
86,Ahmedabad,2024-01-09,35.7,333,24.0,294.0,0.8
96,Ahmedabad,2024-01-10,21.1,187,35.7,333.0,0.2


In [12]:
import importlib
import integration_adapter

importlib.reload(integration_adapter)

<module 'integration_adapter' from 'C:\\Users\\PC\\NICAI_project\\integration_adapter.py'>

In [13]:
from integration_adapter import run_engine

test = {
    "temperature": 35,
    "aqi": 220,
    "trend": 0.8,
    "city": "Mumbai"
}

print(run_engine(test))

{'risk_level': 'HIGH', 'anomaly_type': 'high_pollution', 'temporal_context': 'RISING', 'spatial_context': 'Mumbai', 'confidence': 'HIGH', 'explanation': 'high_pollution detected in Mumbai with RISING trend due to high pollution levels and elevated temperature.', 'recommendation_signal': 'high_pollution'}
